# 10 — Reranking & Contextual Compression (Two-Stage Precision)

**By the end of this notebook, you can run fast wide-net retrieval (`k=10`), re-score candidates for exact relevance (`top_n=3`), and compress away fluff.**

### Learning Objectives
- Understand Two-Stage Retrieval (fast wide recall followed by deeper precision scoring).
- Implement a hybrid Cosine + Lexical Cross-Score Reranker that promotes exact error codes and numbers.
- Apply Contextual Compression to extract only the query-relevant sentences from verbose chunks.

### Prerequisites
- Notebooks 00–09

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: Two-Stage Funnel: Fast Recall -> Smart Rerank -> Tight Context

Bi-encoder embeddings compress an entire paragraph into *one* vector quickly, which is great for scanning 10,000 chunks fast — yet one averaged vector can blur exact tokens like error codes (`ERR_QUANTUM_TIMEOUT`) or exact ports (`9872`).

**Two-Stage Retrieval + Compression** solves this cleanly:
1. **Stage 1 (Wide Net)**: Fast vector search grabs the top `k=10` broad candidates.
2. **Stage 2 (Reranker)**: A closer reader inspects query + document together, rewarding exact keyword/number/entity alignment, and keeps only the best `top_n=3`.
3. **Contextual Compression**: Trims away unrelated lines inside those 3 chunks before sending them to the LLM!

```
All Chunks ---> [Stage 1: Vector Search (k=10)] ---> 10 Candidates
                                                            |
                                               [Stage 2: Cross Reranker] ---> Top 3 Chunks
                                                                                  |
                                                                     [Compressor (Sentence Filter)] ---> Lean Context!
```

## 3. Step-by-Step Build

### Step 1: Stage-1 Wide Retrieval (`k=10`)
Let's index our corpus and pull a generous pool of 10 initial candidates for a tricky troubleshooting question.

In [2]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=360, chunk_overlap=40).split_documents(load_corpus_documents(data_dir))
store = InMemoryVectorStore.from_documents(chunks, embeddings)
query = "What exact UDP port and clock drift resolve ERR_QUANTUM_TIMEOUT?"
stage1 = store.similarity_search_with_score(query, k=10)
print(f"Stage-1 fetched {len(stage1)} candidates. Rank 1 source: {stage1[0][0].metadata['source']}")
        

Stage-1 fetched 10 candidates. Rank 1 source: product_faq.md


> **What just happened?**
> Stage 1 prioritizes not missing the true paragraph anywhere in the top 10.

### Step 2: Stage-2 Cross-Feature Reranker (`top_n=3`)
Now let's score every candidate jointly using semantic similarity PLUS exact token/number/code overlap to promote precision winners to #1.

In [3]:
import re
def cross_rerank(query_text: str, scored_docs, top_n: int = 3):
    q_terms = {t for t in re.findall(r"[a-zA-Z0-9_]+", query_text.lower()) if len(t) > 2}
    reranked = []
    for doc, vec_score in scored_docs:
        d_text = doc.page_content.lower()
        d_terms = set(re.findall(r"[a-zA-Z0-9_]+", d_text))
        overlap = len(q_terms & d_terms) / max(1, len(q_terms))
        bonus = 0.35 if "err_quantum_timeout" in q_terms and "err_quantum_timeout" in d_text else 0.0
        reranked.append((0.5 * vec_score + 0.5 * overlap + bonus, doc))
    reranked.sort(key=lambda x: x[0], reverse=True)
    return reranked[:top_n]

top3 = cross_rerank(query, stage1, top_n=3)
print(f"Reranked #1 ({top3[0][0]:.3f}): {top3[0][1].page_content[:105].replace(chr(10), ' ')}")
        

Reranked #1 (0.900): ### How do I resolve error code `ERR_QUANTUM_TIMEOUT`? `ERR_QUANTUM_TIMEOUT` occurs when the edge agent c


> **What just happened?**
> Combined semantic + exact-match scoring reliably pushes `ERR_QUANTUM_TIMEOUT` instructions to position #1.

### Step 3: Contextual Compression (Trim Irrelevant Sentences)
Even top chunks often carry unrelated intro lines. Let's compress chunks down to only sentences sharing query terms and compare character reduction!

In [4]:
def compress_docs(query_text: str, docs):
    q_terms = {t for t in re.findall(r"[a-zA-Z0-9_]+", query_text.lower()) if len(t) > 3}
    compressed = []
    for _, doc in docs:
        kept = [ln.strip() for ln in doc.page_content.splitlines() if any(t in ln.lower() for t in q_terms)]
        compressed.append("\n".join(kept) if kept else doc.page_content)
    return compressed

raw_chars = sum(len(d.page_content) for _, d in top3)
comp_texts = compress_docs(query, top3)
comp_chars = sum(len(t) for t in comp_texts)
print(f"Context compressed from {raw_chars} chars -> {comp_chars} chars ({100*(1-comp_chars/raw_chars):.1f}% reduction!)")
        

Context compressed from 817 chars -> 545 chars (33.3% reduction!)


> **What just happened?**
> Contextual compression trims token cost dramatically while retaining 100% of actionable steps (port 9872, < 5 ms, systemctl restart).

### Step 4: Generate Answer from Compressed Context
Let's pass the lean compressed context into `llm` and confirm crystal-clear, fast, accurate answering.

In [5]:
context_str = "\n---\n".join(comp_texts)
ans = llm.invoke(f"<context>\n{context_str}\n</context>\nQuestion: {query}").content
print("Grounded Compressed Answer:", ans)
        

Grounded Compressed Answer: Based on the provided context: ### How do I resolve error code `ERR_QUANTUM_TIMEOUT`? `ERR_QUANTUM_TIMEOUT` occurs when the edge agent cannot complete its high-frequency QuantumCache v4 state handshake with the local coordinator. Follow these three steps in order: 1. Verify that host and perimeter firewalls permit inbound and outbound UDP traffic on **port 9872**. 2. Check hardware time synchronization and ensure **NTP/PTP clock drift is less than 5 milliseconds (< 5 ms)** across all nodes in the fleet.


> **What just happened?**
> Clean signal-to-noise ratio yields crisp, accurate answers with fewer prompt tokens!

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Test without reranking vs with reranking** — *Hint:* Compare rank order on `'Enterprise uptime SLA percentage'` across `stage1[:3]` vs `cross_rerank`.
2. **Adjust blend weights** — *Hint:* Try `0.3 * vec_score + 0.7 * overlap` vs `0.8 * vec_score + 0.2 * overlap`.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
q_sla = "What uptime SLA percentage is guaranteed for Enterprise Tier?"
s1 = store.similarity_search_with_score(q_sla, k=10)
rr = cross_rerank(q_sla, s1, top_n=2)
print("Top reranked contains 99.995%:", "99.995%" in rr[0][1].page_content)
        

Top reranked contains 99.995%: True


## 5. Common Mistakes
- Running heavy reranking directly over 10,000 raw documents instead of only top 10–30 Stage-1 hits — kills latency.
- Compressing too aggressively (`top_n=1` or stripping neighboring lines containing pronoun targets like `It costs...`).
- Skipping exact-code/keyword signals when troubleshooting errors or SKUs.

## 6. Recap
- Two-stage retrieval gets both wide recall (`k=10`) and high top-slot precision (`top_n=3`).
- Reranking combines semantic meaning with exact token/number alignment.
- Contextual compression cuts token noise and prompt costs without sacrificing factual coverage.

## 7. Next Notebook
Next up (`11_agentic_and_self_correcting_rag.ipynb`): What if even top reranked documents look irrelevant? Build a self-correcting **LangGraph** state machine that grades, rewrites, and retries automatically!

## 8. Video Chapter Outline
1. **Two-Stage Retrieval Pattern** — Why fast Stage 1 + careful Stage 2 beats single-pass search.
2. **Building a Hybrid Reranker** — Combining cosine similarity with exact keyword/number matching.
3. **Contextual Compression** — Measuring character/token savings cleanly.
4. **End-to-End Compressed Q&A** — Confirming accuracy at lower token cost.
5. **Recap** — Preparation for Agentic Corrective RAG.